In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import pandas as pd
import requests
import numpy as np

In [3]:
def is_number(s):
    try:
        float(s)   # works for int, float, "NaN", "inf"
        return True
    except ValueError:
        return False


def HE_find_nutrition(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    try:
        name = soup.find("h1", class_ = "product-name").text
    except:
        name = "Name not found"
    try:
        div = soup.find("div", class_="product-information")
    except:
        print("Nutritional information not found for: " + name)
        return
    try:
        table = div.find("table")
    except:
        return
    # --- 1. Get table ---
    if not table:
        print("No table found")
        return None

    # --- 2. Extract column headers from thead ---
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    if len(headers) == 0:
        try:
            # fallback: use first row as headers
            first_row = table.find("tr")
            headers = [td.get_text(strip=True) for td in first_row.find_all("td")]
        except:
            print("Empty table")
            return None

    # remove empty headers
    headers = [h if h else headers[0] for h in headers]

    # --- 3. Extract rows ---
    rows = []
    row_headers = []
    last_row_header = None

    for tr in table.find("tbody").find_all("tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]

        # If the first cell is blank, inherit previous header
        if cells and cells[0] == "" and last_row_header is not None:
            row_headers.append(last_row_header)
        else:
            row_headers.append(cells[0])
            last_row_header = cells[0] if cells else last_row_header

        # Pad the row to match header length
        if len(cells) < len(headers):
            cells += [""] * (len(headers) - len(cells))

        rows.append(cells)

    # --- 4. Build DataFrame ---
    df = pd.DataFrame(rows, columns=headers, index=row_headers)
    df.index.name = None

    # --- 5. Drop the first column (duplicate of index) ---
    if df.columns[0].strip().lower() == df.index.name or True:
        df = df.iloc[:, 1:]

    return df

In [2]:
# global list to store nutrient names in discovery order
all_HE_nutrients = []

def HE_collect_nutrients(html_text):
    global all_HE_nutrients
    soup = BeautifulSoup(html_text, 'html.parser')
    
    # --- 1. Get product name ---
    try:
        name = soup.find("h1", class_="product-name").get_text(strip=True)
    except:
        name = "Name not found"
    
    # --- 2. Find nutritional info section ---
    div = soup.find("div", class_="product-information")
    if not div:
        print("⚠️ Nutritional information not found for:", name)
        return
    
    table = div.find("table")
    if not table:
        print("⚠️ No table found for:", name)
        return
    
    # --- 3. Extract rows and row headers ---
    row_headers = []
    last_row_header = None
    
    tbody = table.find("tbody")
    if not tbody:
        print("⚠️ No <tbody> found for:", name)
        return
    
    for tr in tbody.find_all("tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if not cells:
            continue
        
        # Handle merged cells or repeated categories
        if cells[0] == "" and last_row_header is not None:
            row_headers.append(last_row_header)
        else:
            row_headers.append(cells[0])
            last_row_header = cells[0]
    
    # --- 4. Add to global list if new ---
    for nutrient in row_headers:
        if nutrient and nutrient not in all_HE_nutrients:
            all_HE_nutrients.append(nutrient)
    
    print(f"{name}: found {len(row_headers)} nutrients")



In [7]:
link = "https://www.healthspanelite.co.uk/elite-all-blacks-ultimate-whey-protein-blend-chocolate/"
html_text = requests.get(link).text

In [27]:
df = HE_find_nutrition(html_text)
df

,Quantity per 100g,Quantity per 37.5g serving
Energy,1633 kJ,612 kJ
Energy,390 kcal,146 kcal
Fat,6.6 g,2.5 g
of which Saturates,2.0 g,0.8 g
Carbohydrate,21 g,7.9 g
of which Sugars,1.0 g,0.4 g
Fibre,4.4 g,1.7 g
Protein,63 g,24 g
Salt,2.2 g,0.82 g
Sodium,879 mg,330 mg


In [ ]:
URL = "https://www.healthspanelite.co.uk/protein/"
list_page = requests.get(URL)
soup_product_list_page = BeautifulSoup(list_page.text,'html.parser')
product_list = soup_product_list_page.find("div", class_="product-list")
URL_prefix = "https://www.healthspanelite.co.uk/"

for product_link_tag  in product_list.find_all("a", class_=lambda c: c is None or "quick-add-layer-trigger" not in c):
    product_href = URL_prefix + product_link_tag.get("href")
    print("Product page URL:", product_href)
    html_text = requests.get(product_href).text
    df = HE_find_nutrition(html_text)
    print(df)

Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-ultimate-whey-protein-blend-chocolate/
                           Quantity per 100 g Quantity per 37.5 g serving
Energy                                1572 kJ                      578 kJ
Energy                               369 kcal                    138 kcal
Fat                                     4.5 g                       1.7 g
of which Saturates                      3.1 g                       1.2 g
Carbohydrate                             15 g                       5.7 g
of which Sugars                         3.4 g                       1.3 g
Fibre                                   1.5 g                       0.6 g
Protein                                  65 g                        24 g
Salt                                    0.7 g                      0.25 g
Sodium                                 266 mg                      100 mg
Actazin Kiwi Fruit Powder                                          300 mg
pro

In [33]:
URL = "https://www.healthspanelite.co.uk/sports-nutrition/"
list_page = requests.get(URL)
soup_product_list_page = BeautifulSoup(list_page.text,'html.parser')
product_list = soup_product_list_page.find("div", class_="product-list")
URL_prefix = "https://www.healthspanelite.co.uk/"

for product_link_tag  in product_list.find_all("a", class_=lambda c: c is None or "quick-add-layer-trigger" not in c):
    product_href = URL_prefix + product_link_tag.get("href")
    print("Product page URL:", product_href)
    html_text = requests.get(product_href).text
    df = HE_find_nutrition(html_text)
    print(df)

Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-creatine-monohydrate-unflavoured/
                                 Per serving suggestion
Unflavoured Creatine Monohydrate                    5 g
Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-creatine-monohydrate-unflavoured/
                                 Per serving suggestion
Unflavoured Creatine Monohydrate                    5 g
Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-pre-workout-fuel-mixed-berry/
                                                   Per 100 g  \
Energy                                               1643 kJ   
Energy                                              387 kcal   
Fat                                                    0.1 g   
of which Saturates                                    <0.1 g   
Carbohydrate                                            64 g   
of which Sugars                                        3.7 g   
Protein             

In [32]:
URL = "https://www.healthspanelite.co.uk/vitamins-and-supplements/"
list_page = requests.get(URL)
soup_product_list_page = BeautifulSoup(list_page.text,'html.parser')
product_list = soup_product_list_page.find("div", class_="product-list")
URL_prefix = "https://www.healthspanelite.co.uk/"

for product_link_tag  in product_list.find_all("a", class_=lambda c: c is None or "quick-add-layer-trigger" not in c):
    product_href = URL_prefix + product_link_tag.get("href")
    print("Product page URL:", product_href)
    html_text = requests.get(product_href).text
    df = HE_find_nutrition(html_text)
    print(df)

Product page URL: https://www.healthspanelite.co.uk//elite-magnesium-plus/
                                                   Per tablet % NRV
Magnesium                                              375 mg   100
Vitamin B1                                           0.275 mg    25
Vitamin B2                                            0.35 mg    25
Niacin                                                4 mg NE    25
Pantothenic Acid                                       1.5 mg    25
Vitamin B6                                             0.7 mg    50
Folic Acid                                              50 µg    25
Vitamin B12                                           0.63 µg    25
Biotin                                                  25 µg    50
Vitamin C                                               40 mg    50
NRV = Nutrient Reference ValueNE = Niacin Equiv...                 
Product page URL: https://www.healthspanelite.co.uk//elite-magnesium-plus/
                                  

In [4]:
URL = "https://www.healthspanelite.co.uk/protein/"
list_page = requests.get(URL)
soup_product_list_page = BeautifulSoup(list_page.text,'html.parser')
product_list = soup_product_list_page.find("div", class_="product-list")
URL_prefix = "https://www.healthspanelite.co.uk/"

for product_link_tag  in product_list.find_all("a", class_=lambda c: c is None or "quick-add-layer-trigger" not in c):
    product_href = URL_prefix + product_link_tag.get("href")
    print("Product page URL:", product_href)
    html_text = requests.get(product_href).text
    df = HE_collect_nutrients(html_text)
URL = "https://www.healthspanelite.co.uk/sports-nutrition/"
list_page = requests.get(URL)
soup_product_list_page = BeautifulSoup(list_page.text,'html.parser')
product_list = soup_product_list_page.find("div", class_="product-list")
URL_prefix = "https://www.healthspanelite.co.uk/"

for product_link_tag  in product_list.find_all("a", class_=lambda c: c is None or "quick-add-layer-trigger" not in c):
    product_href = URL_prefix + product_link_tag.get("href")
    print("Product page URL:", product_href)
    html_text = requests.get(product_href).text
    df = HE_collect_nutrients(html_text)
URL = "https://www.healthspanelite.co.uk/vitamins-and-supplements/"
list_page = requests.get(URL)
soup_product_list_page = BeautifulSoup(list_page.text,'html.parser')
product_list = soup_product_list_page.find("div", class_="product-list")
URL_prefix = "https://www.healthspanelite.co.uk/"

for product_link_tag  in product_list.find_all("a", class_=lambda c: c is None or "quick-add-layer-trigger" not in c):
    product_href = URL_prefix + product_link_tag.get("href")
    print("Product page URL:", product_href)
    html_text = requests.get(product_href).text
    df = HE_collect_nutrients(html_text)

print(sorted(all_HE_nutrients))

Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-ultimate-whey-protein-blend-chocolate/
All Blacks Ultimate Whey Protein Blend − Chocolate: found 13 nutrients
Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-ultimate-whey-protein-blend-chocolate/
All Blacks Ultimate Whey Protein Blend − Chocolate: found 13 nutrients
Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-ultimate-whey-protein-blend-vanilla/
All Blacks Ultimate Whey Protein Blend − Vanilla: found 13 nutrients
Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-ultimate-whey-protein-blend-vanilla/
All Blacks Ultimate Whey Protein Blend − Vanilla: found 13 nutrients
Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-plant-protein-vegan-blend-vanilla/
All Blacks Plant Protein Vegan Blend − Vanilla: found 14 nutrients
Product page URL: https://www.healthspanelite.co.uk//elite-all-blacks-plant-protein-vegan-blend-vanilla/
All Blacks 